In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import tiktoken

print("PyTorch version:", torch.__version__)
print("Pandas version:", pd.__version__)
print("tiktoken imported successfully!")

PyTorch version: 2.14.1+cpu
Pandas version: 3.0.3
tiktoken imported successfully!


In [2]:
# Load the same IMDB dataset used in Activity 1
df = pd.read_csv("../Activity_1/IMDB Dataset.csv")

print("Original dataset shape:", df.shape)

# Use the same 2,500-row sample as Activity 1
df = df.sample(n=2500, random_state=42).reset_index(drop=True)

print("Sampled dataset shape:", df.shape)

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Original dataset shape: (50000, 2)
Sampled dataset shape: (2500, 2)

Sentiment distribution:
sentiment
positive    1261
negative    1239
Name: count, dtype: int64


In [3]:
# Task 1: Create numeric labels
# positive = 1
# negative = 0

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

# Shuffle the dataset using the same random state
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 80% training, 20% testing
split_index = int(0.8 * len(df))

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Extract reviews and labels
X_train = train_df["review"].tolist()
y_train = train_df["label"].tolist()

X_test = test_df["review"].tolist()
y_test = test_df["label"].tolist()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining labels:")
print(pd.Series(y_train).value_counts())

print("\nTesting labels:")
print(pd.Series(y_test).value_counts())

Training samples: 2000
Testing samples: 500

Training labels:
1    1014
0     986
Name: count, dtype: int64

Testing labels:
0    253
1    247
Name: count, dtype: int64


In [4]:
# Task 1 & 2: Tokenization and padding

# Initialize the same tokenizer used in Activity 1
enc = tiktoken.get_encoding("cl100k_base")

# Same maximum sequence length as Activity 1
MAX_LEN = 200


def tokenize_and_pad(text):
    # Convert text into token IDs
    tokens = enc.encode(text)

    # Truncate longer sequences
    tokens = tokens[:MAX_LEN]

    # Pad shorter sequences with 0
    if len(tokens) < MAX_LEN:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    return tokens


# Tokenize training and testing reviews
X_train_tokens = [tokenize_and_pad(review) for review in X_train]
X_test_tokens = [tokenize_and_pad(review) for review in X_test]

print("Tokenizer:", enc.name)
print("Training sequences:", len(X_train_tokens))
print("Testing sequences:", len(X_test_tokens))
print("Sequence length:", len(X_train_tokens[0]))

print("\nFirst 20 tokens:")
print(X_train_tokens[0][:20])

Tokenizer: cl100k_base
Training sequences: 2000
Testing sequences: 500
Sequence length: 200

First 20 tokens:
[3947, 1288, 387, 264, 6037, 430, 5415, 5115, 9539, 430, 9698, 1093, 60776, 34819, 527, 10171, 311, 387, 1903, 304]


In [5]:
# Task 3: Convert tokenized data into PyTorch tensors

X_train_tensor = torch.tensor(X_train_tokens, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_tokens, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("Training input shape:", X_train_tensor.shape)
print("Testing input shape:", X_test_tensor.shape)
print("Training labels shape:", y_train_tensor.shape)
print("Testing labels shape:", y_test_tensor.shape)

print("\nData types:")
print("Input:", X_train_tensor.dtype)
print("Labels:", y_train_tensor.dtype)

Training input shape: torch.Size([2000, 200])
Testing input shape: torch.Size([500, 200])
Training labels shape: torch.Size([2000])
Testing labels shape: torch.Size([500])

Data types:
Input: torch.int64
Labels: torch.int64


In [6]:
# Task 4: Build the LSTM text classification model

class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        # Convert token IDs into dense vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Final classification layer
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # Embedding
        embedded = self.embedding(x)

        # LSTM
        output, (hidden, cell) = self.lstm(embedded)

        # Take the output from the last time step
        last_output = output[:, -1, :]

        # Classification
        logits = self.fc(last_output)

        return logits


# Same settings as Activity 1
vocab_size = enc.n_vocab
embedding_dim = 64
hidden_dim = 64
num_classes = 2

model = LSTMTextClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
)

print(model)

LSTMTextClassifier(
  (embedding): Embedding(100277, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [7]:
# Task 5: Prepare DataLoader, loss function, and optimizer

from torch.utils.data import TensorDataset, DataLoader

# Create training dataset
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

# Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Batch size:", 32)
print("Number of training batches:", len(train_loader))
print("Loss function:", criterion)
print("Optimizer:", optimizer)

Batch size: 32
Number of training batches: 63
Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [8]:
# Task 6: Train the LSTM model

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_loss:.4f}")

Epoch 1/5, Loss: 0.6951
Epoch 2/5, Loss: 0.6801
Epoch 3/5, Loss: 0.6537
Epoch 4/5, Loss: 0.5989
Epoch 5/5, Loss: 0.5262


In [9]:
# Task 7: Evaluate the LSTM model

model.eval()

correct = 0
total = 0

with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1)

    correct = (predictions == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)

accuracy = 100 * correct / total

print(f"Correct predictions: {correct}/{total}")
print(f"LSTM Test Accuracy: {accuracy:.2f}%")

Correct predictions: 253/500
LSTM Test Accuracy: 50.60%


In [10]:
# Task 8: Test the LSTM model on custom reviews

custom_reviews = [
    "This movie was absolutely fantastic! The story was amazing and the actors were excellent.",
    "I really disliked this movie. It was boring, slow, and a complete waste of time.",
    "The movie was okay, but the story was not very interesting."
]

# Tokenize and pad the custom reviews
custom_tokens = [tokenize_and_pad(review) for review in custom_reviews]

# Convert to tensor
custom_tensor = torch.tensor(custom_tokens, dtype=torch.long)

# Predict
model.eval()

with torch.no_grad():
    outputs = model(custom_tensor)
    predictions = torch.argmax(outputs, dim=1)

# Display predictions
for review, prediction in zip(custom_reviews, predictions):
    label = "Positive" if prediction.item() == 1 else "Negative"
    print("Review:", review)
    print("LSTM Prediction:", label)
    print()

Review: This movie was absolutely fantastic! The story was amazing and the actors were excellent.
LSTM Prediction: Negative

Review: I really disliked this movie. It was boring, slow, and a complete waste of time.
LSTM Prediction: Negative

Review: The movie was okay, but the story was not very interesting.
LSTM Prediction: Negative



# Results and Comparison: RNN vs LSTM

## 1. Which model has higher test accuracy?

The **RNN model** achieved higher test accuracy than the LSTM model.

- **RNN Test Accuracy:** 53.60%
- **LSTM Test Accuracy:** 50.60%

Therefore, the RNN performed slightly better on the test dataset in this experiment.

---

## 2. Did LSTM reach lower training loss or in fewer epochs?

Yes. The LSTM reached a lower training loss after 5 epochs.

| Epoch | RNN Loss | LSTM Loss |
|------:|---------:|----------:|
| 1 | 0.7040 | 0.6951 |
| 2 | 0.6704 | 0.6801 |
| 3 | 0.6452 | 0.6537 |
| 4 | 0.6080 | 0.5989 |
| 5 | 0.5698 | **0.5262** |

The LSTM achieved a final training loss of **0.5262**, while the RNN achieved **0.5698**.

However, the lower training loss of the LSTM did not result in higher test accuracy.

---

## 3. Did RNN and LSTM agree on custom predictions? Which seems more reliable?

Yes. Both models gave the same predictions for all three custom reviews.

| Custom Review | RNN | LSTM |
|---|---|---|
| Fantastic movie, amazing story and excellent actors | Negative | Negative |
| Boring, slow and a waste of time | Negative | Negative |
| Story was not very interesting | Negative | Negative |

Both models predicted all three reviews as **Negative**.

The first review is clearly positive, so both models made an incorrect prediction for that review. Therefore, the custom predictions should not be considered highly reliable.

Based on the test accuracy, the **RNN appears slightly more reliable** in this experiment because it achieved 53.60% accuracy compared with 50.60% for the LSTM.

---

## 4. What is LSTM doing differently that might explain the differences?

An ordinary RNN processes the input sequence step by step and can have difficulty remembering information from earlier parts of a long sequence because of the **vanishing gradient problem**.

An LSTM (Long Short-Term Memory) uses a **cell state and gates** to control information flow. The main gates are:

- **Forget gate:** Decides what information should be removed.
- **Input gate:** Decides what new information should be stored.
- **Output gate:** Decides what information should be used as output.

These mechanisms help LSTM retain important information for longer sequences.

In this experiment, however, the LSTM achieved a lower training loss but a slightly lower test accuracy. This may be because of the small dataset, simple model architecture, limited number of training epochs, and the characteristics of the dataset.

---

# Conclusion

In this experiment, the **RNN achieved 53.60% test accuracy**, while the **LSTM achieved 50.60%**. Although the LSTM achieved a lower final training loss (**0.5262** compared with **0.5698** for the RNN), the RNN generalized slightly better to the test data.

Both models predicted all three custom reviews as negative. The results show that a lower training loss does not always mean better test performance. LSTM provides better mechanisms for remembering long-term information, but its advantage was not observed in test accuracy in this particular experiment.